In [1]:
import plotly.graph_objects as go
from nilearn import datasets, surface, image
from skimage import measure
from scipy import ndimage
import numpy as np
import os


# ============================================================
# ROI paths
# ============================================================
roi_paths = {
    # 'GIST': r'C:\Users\yujunchen\OneDrive - University of Florida\Searchlight results\main\tmap_gist_norm_p001.nii.gz',
    # 'Scene': r'C:\Users\yujunchen\OneDrive - University of Florida\Searchlight results\main\tmap_clipvit_p001_fisherz_fdr01.nii.gz',
    # 'Emotion': r'C:\Users\yujunchen\OneDrive - University of Florida\Searchlight results\main\tmap_beh60_p05_v2.nii.gz'
    
    'GIST': r'N:\Experimental_Data\yujunchen\projects\IAPS_Searchlight\outputs\tBrainmap\final_results\tmap_gist_norm_p001.nii.gz',
    'Scene': r'N:\Experimental_Data\yujunchen\projects\IAPS_Searchlight\outputs\tBrainmap\final_results\tmap_clipvit_p001_fisherz_fdr01.nii.gz',
    'Emotion': r'N:\Experimental_Data\yujunchen\projects\IAPS_Searchlight\outputs\tBrainmap\final_results\tmap_beh60_p05_v2.nii.gz'
}

output_dir = r"N:\Experimental_Data\yujunchen\projects\IAPS_Searchlight\visualization"


# ============================================================
# Colors
# ============================================================
roi_colors = {
    'GIST':   '#3498db',
    'Scene':  '#e74c3c',
    'Emotion':'#2ecc71'
}

brain_color = 'lightgray'


# ============================================================
# Helper: mesh from volume
# ============================================================
def create_mesh_from_volume(volume_data, affine, threshold=0.5):

    verts, faces, normals, values = measure.marching_cubes(volume_data, level=threshold)

    verts_h = np.c_[verts, np.ones(len(verts))]
    verts_world = verts_h.dot(affine.T)[:, :3]

    return verts_world, faces


# ============================================================
# Build ROI meshes
# ============================================================
roi_meshes = {}

for roi_name, roi_path in roi_paths.items():

    roi_img = image.load_img(roi_path)
    roi_data = roi_img.get_fdata()

    roi_data_smooth = ndimage.gaussian_filter(roi_data, sigma=0.8)

    verts, faces = create_mesh_from_volume(
        roi_data_smooth,
        roi_img.affine,
        threshold=0.5
    )

    roi_meshes[roi_name] = (verts, faces)


# ============================================================
# Load fsaverage surface
# ============================================================
fsaverage = datasets.fetch_surf_fsaverage()


# ============================================================
# Create figure
# ============================================================
fig = go.Figure()


# ============================================================
# Add brain surface
# ============================================================
brain_trace_ids = []

for hemi in ['left','right']:

    coords, faces = surface.load_surf_mesh(fsaverage[f'pial_{hemi}'])

    fig.add_trace(go.Mesh3d(
        x=coords[:,0],
        y=coords[:,1],
        z=coords[:,2],

        i=faces[:,0],
        j=faces[:,1],
        k=faces[:,2],

        color=brain_color,
        opacity=1.0,

        name=f"Brain {hemi}",
        showlegend=False,
        hoverinfo='skip',

        lighting=dict(
            ambient=0.8,
            diffuse=0.5,
            specular=0.1,
            roughness=0.8
        ),

        lightposition=dict(x=200,y=200,z=200)
    ))

    brain_trace_ids.append(len(fig.data)-1)


# ============================================================
# Add ROI meshes
# ============================================================
roi_trace_ids = {}

for roi_name,(verts,faces) in roi_meshes.items():

    fig.add_trace(go.Mesh3d(

        x=verts[:,0],
        y=verts[:,1],
        z=verts[:,2],

        i=faces[:,0],
        j=faces[:,1],
        k=faces[:,2],

        color=roi_colors[roi_name],
        opacity=0.45,

        name=roi_name,
        showlegend=True,

        lighting=dict(
            ambient=0.7,
            diffuse=0.8,
            specular=0.2,
            roughness=0.6
        ),

        lightposition=dict(x=200,y=200,z=200)

    ))

    roi_trace_ids[roi_name] = len(fig.data)-1


mesh_traces = list(range(len(fig.data)))


# ============================================================
# SLIDERS
# ============================================================

def slider_steps(param,values):

    steps=[]

    for v in values:

        steps.append(dict(
            method="restyle",
            args=[{f"lighting.{param}":v},mesh_traces],
            label=str(v)
        ))

    return steps


ambient_vals=np.round(np.linspace(0.1,1,10),2)
diffuse_vals=np.round(np.linspace(0,1,11),2)
spec_vals=np.round(np.linspace(0,1,11),2)
rough_vals=np.round(np.linspace(0.1,1,10),2)
fres_vals=np.round(np.linspace(0,5,11),2)


# Brain opacity slider
brain_opacity_vals=np.round(np.linspace(0.1,1,10),2)

brain_opacity_steps=[]

for v in brain_opacity_vals:

    brain_opacity_steps.append(dict(
        method="restyle",
        args=[{"opacity":v},brain_trace_ids],
        label=str(v)
    ))


# ROI opacity slider
roi_opacity_vals=np.round(np.linspace(0.1,1,10),2)

roi_opacity_steps=[]

for v in roi_opacity_vals:

    roi_opacity_steps.append(dict(
        method="restyle",
        args=[{"opacity":v},list(roi_trace_ids.values())],
        label=str(v)
    ))


# ============================================================
# COLOR PRESETS
# ============================================================

color_buttons=[]

palette=[
("#3498db","#e74c3c","#2ecc71"),
("#1abc9c","#e67e22","#9b59b6"),
("#2980b9","#c0392b","#27ae60")
]

for i,p in enumerate(palette):

    updates=[]

    for roi,c in zip(['GIST','Scene','Emotion'],p):

        updates.append(dict(
            method="restyle",
            args=[{"color":c},[roi_trace_ids[roi]]]
        ))

    color_buttons.append(dict(
        label=f"Palette {i+1}",
        method="update",
        args=[{},{}]
    ))


# ============================================================
# LIGHT DIRECTION
# ============================================================

light_buttons=[]

lights=[
("Top Front Right",dict(x=200,y=200,z=200)),
("Left",dict(x=-200,y=0,z=100)),
("Right",dict(x=200,y=0,z=100)),
("Front",dict(x=0,y=200,z=100)),
("Top",dict(x=0,y=0,z=300))
]

for name,pos in lights:

    light_buttons.append(dict(
        label=name,
        method="restyle",
        args=[{"lightposition":pos},mesh_traces]
    ))


# ============================================================
# Layout
# ============================================================

fig.update_layout(

    title="GIST (Blue) | Scene (Red) | Emotion (Green) - 3D Brain",

    scene=dict(
        xaxis=dict(visible=False),
        yaxis=dict(visible=False),
        zaxis=dict(visible=False),
        bgcolor='white',
        aspectmode='data'
    ),

    width=2500,
    height=1500,

    paper_bgcolor='white',

    # increase right margin to create a control panel
    margin=dict(t=120, r=350),

    sliders=[

        dict(
            x=1.05,
            y=0.95,
            len=0.25,
            currentvalue=dict(prefix="Ambient "),
            steps=slider_steps("ambient", ambient_vals)
        ),

        dict(
            x=1.05,
            y=0.88,
            len=0.25,
            currentvalue=dict(prefix="Diffuse "),
            steps=slider_steps("diffuse", diffuse_vals)
        ),

        dict(
            x=1.05,
            y=0.81,
            len=0.25,
            currentvalue=dict(prefix="Specular "),
            steps=slider_steps("specular", spec_vals)
        ),

        dict(
            x=1.05,
            y=0.74,
            len=0.25,
            currentvalue=dict(prefix="Roughness "),
            steps=slider_steps("roughness", rough_vals)
        ),

        dict(
            x=1.05,
            y=0.67,
            len=0.25,
            currentvalue=dict(prefix="Fresnel "),
            steps=slider_steps("fresnel", fres_vals)
        ),

        dict(
            x=1.05,
            y=0.60,
            len=0.25,
            currentvalue=dict(prefix="Brain Opacity "),
            steps=brain_opacity_steps
        ),

        dict(
            x=1.05,
            y=0.53,
            len=0.25,
            currentvalue=dict(prefix="ROI Opacity "),
            steps=roi_opacity_steps
        ),
    ],

    updatemenus=[

        dict(
            x=1.05,
            y=1.02,
            xanchor="left",
            yanchor="top",
            buttons=light_buttons,
            direction="down",
            showactive=True
        ),

        dict(
            x=1.05,
            y=0.95,
            xanchor="left",
            yanchor="top",
            buttons=color_buttons,
            direction="down",
            showactive=True
        )

    ]
)


# ============================================================
# Save HTML
# ============================================================

out_html=os.path.join(output_dir,"interactive_brain_controls.html")

fig.write_html(out_html)

print("Saved:",out_html)

Saved: N:\Experimental_Data\yujunchen\projects\IAPS_Searchlight\visualization\interactive_brain_controls.html
